In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd

# Khởi tạo
driver = webdriver.Chrome()
driver.get("https://www.facebook.com")

# --- ĐOẠN NÀY BẠN TỰ CODE ĐĂNG NHẬP ---
# Tìm ô email, pass, sendkeys và click login
# Nên dùng time.sleep() để chờ đăng nhập thủ công cho an toàn nếu dính 2FA
time.sleep(30) # Chờ 30s để bạn tự đăng nhập bằng tay cho chắc

In [ ]:
def get_group_info(group_url):
    driver.get(group_url)
    time.sleep(3) # Chờ load
    
    try:
        # 1. Lấy Tên Nhóm (Thường là thẻ h1)
        group_name = driver.find_element(By.TAG_NAME, "h1").text
        
        # 2. Lấy ID (Xử lý chuỗi từ URL)
        # URL dạng: facebook.com/groups/123456/ -> ID là 123456
        group_id = group_url.split("/groups/")[1].split("/")[0]
        
        # 3. Lấy số thành viên
        # Mẹo: Tìm các thẻ span chứa chữ "thành viên" hoặc "members"
        # Cái này cần F12 để lấy Xpath cụ thể
        member_text = driver.find_element(By.XPATH, "//span[contains(text(), 'thành viên')]").text
        
        return {
            "Id": group_id,
            "Label": group_name,
            "Members": member_text
        }
    except:
        return None

In [ ]:
def get_related_groups(group_id):
    # Giả sử URL trang related groups là như này (cần kiểm tra thực tế)
    related_url = f"https://www.facebook.com/groups/{group_id}/about" 
    driver.get(related_url)
    time.sleep(3)
    
    related_ids = []
    
    # Tìm tất cả các thẻ <a> chứa link trỏ về /groups/...
    # Logic: Quét trang About, tìm các link nhóm khác
    elements = driver.find_elements(By.XPATH, "//a[contains(@href, '/groups/')]")
    
    for elem in elements:
        href = elem.get_attribute('href')
        # Lọc xử lý để lấy ID
        if "groups/" in href:
            target_id = href.split("groups/")[1].split("/")[0]
            if target_id != group_id: # Tránh tự trỏ về mình
                related_ids.append(target_id)
                
    # Loại bỏ trùng lặp
    return list(set(related_ids))

In [ ]:
queue = ["123456789"] # ID nhóm hạt giống ban đầu
visited = set()       # Danh sách các nhóm đã cào để không cào lại
nodes_data = []
edges_data = []

while len(nodes_data) < 300 and len(queue) > 0:
    current_id = queue.pop(0) # Lấy nhóm đầu tiên ra xử lý
    
    if current_id in visited:
        continue
        
    url = f"https://www.facebook.com/groups/{current_id}"
    print(f"Đang cào: {url}")
    
    # 1. Lấy thông tin Node
    info = get_group_info(url)
    if info:
        nodes_data.append(info)
        visited.add(current_id)
        
        # 2. Lấy thông tin Edges (Các nhóm liên quan)
        related_list = get_related_groups(current_id)
        
        for target in related_list:
            # Lưu cạnh
            edges_data.append({
                "Source": current_id,
                "Target": target,
                "Type": "Directed"
            })
            
            # Thêm nhóm mới tìm thấy vào hàng đợi để cào tiếp
            if target not in visited and target not in queue:
                queue.append(target)
    
    time.sleep(5) # Quan trọng: Nghỉ để không bị Facebook khóa

# Lưu ra file
pd.DataFrame(nodes_data).to_csv("database/nodes_community.csv", index=False)
pd.DataFrame(edges_data).to_csv("database/edges_community.csv", index=False)